# Multi-Instance Malware Classifier (ARM Zephyr ELF & BIG 2015 PE)
### Models: VGG16 & ResNet50 | Dynamic Multi-Channel Segmentation

This notebook is **100% self-contained** and includes all package source code and configs embedded directly.
Run this notebook concurrently across **3 separate Kaggle accounts / instances**:
- **Instance 1**: Baselines (Raw byte layouts: S1, S2, S3 for VGG16 & ResNet50)
- **Instance 2**: 3-Channel Section Separation (S4_text_rodata_data, S5_text_rodata_data, S5_imgs1024_text_data, S5_imgs1024_text_rodata)
- **Instance 3**: Advanced Multi-Channel (4 & 5 Channels: S4 4-ch, S5 4-ch, S5 5-ch with dynamic conv1 adaptation)

> **Important**: Ensure **GPU T4 x2** (or GPU P100) and **Internet: On** in the right-hand panel under **Settings**.

In [ ]:
# ==========================================================
# 1. SELECT INSTANCE ID (1, 2, or 3)
# ==========================================================
INSTANCE_ID = "1"  # <-- SET TO "1", "2", or "3" FOR EACH KAGGLE INSTANCE

EPOCHS = 20
BATCH_SIZE = 32         # 32 is optimal for Kaggle GPU (P100 / T4)
VAL_SPLIT = 0.2         # 80% train / 20% validation split
DATASET_CONFIG = "arm_zephyr"  # "arm_zephyr" for ARM ELF, "big2015" for PE
USE_PRETRAINED = True   # True: ImageNet transfer learning


In [ ]:
# ==========================================================
# 2. Extract Project Code & Configs (Self-Contained)
# ==========================================================
import os, sys, base64, io, zipfile, glob

CODE_ZIP_B64 = "..."

dest_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
if not os.path.exists(os.path.join(dest_dir, "kaggle_runner.py")):
    print("Extracting codebase and configs to", dest_dir, "...")
    zip_bytes = base64.b64decode(CODE_ZIP_B64)
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        z.extractall(dest_dir)
    print("Code successfully deployed!")
else:
    print("Codebase already present.")

# Ensure local src is on python path and dependencies are ready
src_dir = os.path.join(dest_dir, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
!pip install -q pyelftools


In [ ]:
# ==========================================================
# 3. Environment & Dataset Auto-Detection
# ==========================================================
import torch

print("PyTorch Version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU Model:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected! Please enable GPU accelerator in Kaggle settings (right sidebar).")

# Auto-detect preprocessed NPZ dataset directory under /kaggle/input
data_dir = None
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if any(f.endswith(".npz") for f in files):
            data_dir = os.path.dirname(root)
            break

if not data_dir:
    for cand in ["./Processed_Dataset", "../Processed_Dataset", "./data"]:
        if os.path.isdir(cand) and glob.glob(os.path.join(cand, "**/*.npz"), recursive=True):
            data_dir = cand
            break

if not data_dir:
    raise FileNotFoundError("Dataset not found! Please attach 'arm-malware-npz' via '+ Add Input' in Kaggle.")

sample_count = len(glob.glob(os.path.join(data_dir, "**/*.npz"), recursive=True))
print(f"Found dataset at: {data_dir} ({sample_count} samples)")


In [ ]:
# ==========================================================
# 4. Launch Experiment Instance
# ==========================================================
runner_path = os.path.join(dest_dir, "kaggle_runner.py")
cmd = (
    f"python {runner_path} "
    f"-i {INSTANCE_ID} "
    f"--data-dir {data_dir} "
    f"--dataset {DATASET_CONFIG} "
    f"-e {EPOCHS} "
    f"-b {BATCH_SIZE} "
    f"--val-split {VAL_SPLIT} "
    f"--device {device.type}"
)
if not USE_PRETRAINED:
    cmd += " --no-pretrained"

print("Starting training run:")
print(cmd)
print("=" * 60)
!{cmd}


In [ ]:
# ==========================================================
# 5. Display Plots & Download Results Package
# ==========================================================
from IPython.display import Image, display

artifacts_dir = f"{data_dir}/artifacts" if os.path.isdir(f"{data_dir}/artifacts") else f"{dest_dir}/artifacts"
plots = glob.glob(f"{artifacts_dir}/**/*.png", recursive=True)
print(f"Generated {len(plots)} visualization plots:")
for p in sorted(plots):
    print("-", os.path.basename(p))
    display(Image(filename=p))

zip_name = f"/kaggle/working/results_instance_{INSTANCE_ID}.zip"
!zip -q -r {zip_name} {artifacts_dir}
print(f"
All checkpoints, metadata, and plots are packaged in: {zip_name}")
print("Download this zip directly from the /kaggle/working panel on the right!")
